In [ ]:
import pandas as pd
import numpy as np
import sklearn.preprocessing as sks
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
data=pd.read_csv("insurance.csv")
data
sencoder=sks.OneHotEncoder(sparse_output=False,handle_unknown="ignore")
scaler=sks.StandardScaler()
lr=LinearRegression()
sg=SGDRegressor()
num_cols=['charges','bmi','age']
def process_data(df):
 
    region_encoded = sencoder.fit_transform(df[['region']])
    encoded_cols = sencoder.get_feature_names_out(['region'])
    encoded_df = pd.DataFrame(region_encoded, columns=encoded_cols, index=df.index)
    df = pd.concat([df.drop(columns='region'), encoded_df], axis=1)

    sex_encoded = sencoder.fit_transform(df[['sex']])
    encoded_cols = sencoder.get_feature_names_out(['sex'])
    encoded_df = pd.DataFrame(sex_encoded, columns=encoded_cols, index=df.index)
    df = pd.concat([df.drop(columns='sex'), encoded_df], axis=1)

    smoker_encoded = sencoder.fit_transform(df[['smoker']])
    encoded_cols = sencoder.get_feature_names_out(['smoker'])
    encoded_df = pd.DataFrame(smoker_encoded, columns=encoded_cols, index=df.index)
    df = pd.concat([df.drop(columns='smoker'), encoded_df], axis=1)

    df[num_cols]=scaler.fit_transform(df[num_cols])
    return df

new_data=process_data(data)
x = new_data.drop(['charges'], axis=1)
y = new_data['charges']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
def baseline():
    lr = LinearRegression()
    lr.fit(x_train,y_train)
    y_pred = lr.predict(x_test)
    print(f"Linear Regression R2: {lr.score(x_test, y_test)}")
    params={
        'penalty':[ 'l2' ,'l1', 'elasticnet', None],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate': ['invscaling', 'adaptive'],
        'eta0': [0.01, 0.05]
    }
    sg=SGDRegressor()
    sgd_grid = GridSearchCV(estimator=sg, param_grid=params, cv=5, scoring='r2')
    sgd_grid.fit(x_train, y_train)
    print(f"SGD Score: {sgd_grid.best_score_}")
    dt = DecisionTreeRegressor(random_state=42)
    dt.fit(x_train, y_train)
    print(f"Decision Tree R2: {dt.score(x_test, y_test)}")
baseline()
def ensemble():
    param = {
        'n_estimators': [100, 500, 1000],
        'max_features': ['auto', 'sqrt', 'log2', None],
        'bootstrap': [True, False],
        'max_depth': [10, 11, 110, None],
    }
    rf = RandomForestRegressor()
    rf_grid = GridSearchCV(estimator=rf, param_grid=param, cv=5, scoring='r2')
    rf_grid.fit(x_train, y_train)
    print(f"Random Forest best CV R2: {rf_grid.best_score_}, best params: {rf_grid.best_params_}")



SyntaxError: cannot assign to literal here. Maybe you meant '==' instead of '='? (654742743.py, line 63)